In [1]:
!pip install itables
!pip install chardet
import pandas as pd
from io import StringIO
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

In [2]:
csv_file = './cross-verified-database.csv'

corrected_lines = []  # A list to hold corrected lines
error_log = []  # A list to log errors

with open(csv_file, 'rb') as file:  # Open the file in binary mode to avoid decoding errors
    line_number = 0
    while True:
        line_number += 1
        try:
            line = next(file)
            decoded_line = line.decode('utf-8')  # Attempt to decode each line as UTF-8
            corrected_lines.append(decoded_line)
        except StopIteration:  # If we reach the end of the file
            break
        except UnicodeDecodeError as e:  # Catch decoding errors
            error_log.append((line_number, line))
            corrected_lines.append(line.decode('latin1'))

corrected_csv_content = ''.join(corrected_lines)
csv_content_io = StringIO(corrected_csv_content)
dataframe = pd.read_csv(csv_content_io)
target_headers = ['wikidata_code', 'name','gender','un_region' , 'un_subregion', 'level1_main_occ','level2_main_occ','level3_main_occ','wiki_readers_2015_2018','ranking_visib_5criteria', 'total_noccur_links_b']

In [4]:
actors = dataframe[
    (dataframe['level3_main_occ'].isin(['actor', 'film']))&
    (dataframe['list_wikipedia_editions'].str.contains('enwiki'))
].sort_values(by='ranking_visib_5criteria')[target_headers]

writers = dataframe[
    (dataframe['level3_main_occ'].isin(['writer', 'novelist', 'screenwriter', 'poet'])) &
    (dataframe['list_wikipedia_editions'].str.contains('enwiki'))
].sort_values(by='ranking_visib_5criteria')[target_headers]

painters = dataframe[
    (dataframe['level3_main_occ'].isin(['painter'])) &
    (dataframe['list_wikipedia_editions'].str.contains('enwiki'))
].sort_values(by='ranking_visib_5criteria')[target_headers]

musicians =  dataframe[
    (dataframe['level3_main_occ'].isin([ 'singer', 'composer'])) &
    (dataframe['list_wikipedia_editions'].str.contains('enwiki'))
].sort_values(by='ranking_visib_5criteria')[target_headers]

total_size =   actors.shape[0] + writers.shape[0] + painters.shape[0] + musicians.shape[0]
dataset_size = 500
dataset_1 = pd.concat([
    actors.head(dataset_size * actors.shape[0] // total_size),
    writers.head(dataset_size * writers.shape[0] // total_size +1),
    painters.head(dataset_size * painters.shape[0] // total_size),
    musicians.head(dataset_size * musicians.shape[0] // total_size ),
])
dataset_1.sort_values(by='ranking_visib_5criteria',inplace=True)
dataset_1.to_csv('./1.csv')
dataset_1[target_headers]

In [5]:
dataset_1['level3_main_occ'].value_counts()

In [6]:
actors = dataframe[
    (dataframe['level3_main_occ'].isin(['actor', 'film']))&
    (dataframe['list_wikipedia_editions'].str.contains('enwiki'))
].sort_values(by='ranking_visib_5criteria')[target_headers]
actors.head(500).to_csv('./2.csv')
actors

In [7]:
pd.DataFrame(actors.head(500)['level3_main_occ'].value_counts())

In [8]:
import math

def create_dataset_by_category(categories, dataset_size, gender =['Male','Female']):
    frames={}
    for category in categories:
        frames[category] = dataframe[
        (dataframe['level3_main_occ'].isin([category])) &
        (dataframe['gender'].isin(gender))&
        (dataframe['list_wikipedia_editions'].str.contains('enwiki'))
        ].sort_values(by='ranking_visib_5criteria')[target_headers]
    frequencies = {}
    total_rows = 0
    for category,frame in frames.items():
        frequencies[category] = frame.shape[0]
    total_rows = sum(frequencies.values())
    print(total_rows)
    result = pd.DataFrame()
    for category,frame in frames.items():
        result = pd.concat([result,
            frame.head(round(dataset_size * frame.shape[0] / total_rows)),
        ])
    return result.sort_values(by='ranking_visib_5criteria')[target_headers].head(dataset_size)

In [9]:
categories = ['sport','journalist', 'politician', 'football', 'actor', 'writer', 'painter', 'singer', 'player', 'lawyer']
dataset_3 = create_dataset_by_category(categories,500)
dataset_3.to_csv("./3.csv")

In [10]:
dataset_3['level3_main_occ'].value_counts()

In [11]:
categories = ['sport','journalist', 'politician', 'football', 'actor','writer','painter','singer','player','athletic']
dataset_4 = create_dataset_by_category(categories,500,['Female'])
dataset_4.to_csv('./4.csv')
dataset_4

In [12]:
dataset_4['level3_main_occ'].value_counts()